# Relational Databases — First Contact

Think of Postgres as the reliable workhorse of data engineering: data lives in rows and columns with an enforced schema, every write is protected by ACID guarantees, and the query planner figures out the fastest path to your answer so you don't have to. If you need consistency, joins, and decades of battle-tested reliability, reach for a relational database first.

## What makes relational different

- **ACID** — Atomicity, Consistency, Isolation, Durability: every transaction either fully commits or fully rolls back; no partial writes reach disk.
- **Schema enforcement** — column types and constraints are defined upfront; bad data is rejected at write time, not discovered at query time.
- **Query planner** — you declare *what* you want (SQL); Postgres decides *how* to fetch it (seq scan, index scan, hash join, merge join) based on statistics.
- **When to use** — structured data with known shape, need for joins across entities, strong consistency requirements, reporting and aggregations.

In [2]:
from pathlib import Path
import sys

# _setup is a sibling directory of this notebook.
# Works whether Jupyter is launched from the notebook's folder or the workspace root.
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

import pandas as pd
from db_connections import get_postgres_conn

conn = get_postgres_conn()
print("Connected.")

Connected.


## 5 telemetry queries

In [3]:
# Query 1 — endpoint count by datacenter
q1 = """
SELECT
    datacenter,
    COUNT(*) AS endpoint_count
FROM telemetry.endpoints
GROUP BY datacenter
ORDER BY endpoint_count DESC
"""
df1 = pd.read_sql(q1, conn)
print(f"Datacenters: {len(df1)}")
df1

Datacenters: 4


C:\Users\shareuser\AppData\Local\Temp\ipykernel_27980\3856423289.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(q1, conn)


,datacenter,endpoint_count
0,SNG1,2579
1,NYC1,2489
2,LON1,2485
3,NYC2,2447


In [4]:
# Query 2 — top 10 most alerted endpoints
q2 = """
SELECT
    e.hostname,
    e.datacenter,
    e.service_type,
    COUNT(a.alert_id) AS alert_count
FROM telemetry.endpoints e
JOIN telemetry.alerts a ON a.endpoint_id = e.endpoint_id
GROUP BY e.endpoint_id, e.hostname, e.datacenter, e.service_type
ORDER BY alert_count DESC
LIMIT 10
"""
df2 = pd.read_sql(q2, conn)
print("Top 10 most alerted endpoints:")
df2

Top 10 most alerted endpoints:


C:\Users\shareuser\AppData\Local\Temp\ipykernel_27980\2608023395.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(q2, conn)


,hostname,datacenter,service_type,alert_count
0,srv-04452.citi.internal,NYC1,worker,12
1,srv-07578.citi.internal,NYC1,web,11
2,srv-03423.citi.internal,NYC1,monitor,10
3,srv-01617.citi.internal,SNG1,db,10
4,srv-02253.citi.internal,SNG1,worker,10
5,srv-03610.citi.internal,NYC2,cache,10
6,srv-05519.citi.internal,NYC2,cache,9
7,srv-01153.citi.internal,SNG1,monitor,9
8,srv-07903.citi.internal,SNG1,cache,9
9,srv-07573.citi.internal,LON1,worker,9


In [5]:
# Query 3 — avg CPU percent per service_type, last 7 days
q3 = """
SELECT
    e.service_type,
    ROUND(AVG(m.value)::numeric, 2) AS avg_cpu_percent,
    COUNT(*) AS sample_count
FROM telemetry.metrics m
JOIN telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
WHERE
    m.metric_name = 'cpu_percent'
    AND m.recorded_at >= NOW() - INTERVAL '7 days'
GROUP BY e.service_type
ORDER BY avg_cpu_percent DESC
"""
df3 = pd.read_sql(q3, conn)
print("Avg CPU % by service type (last 7 days):")
df3

Avg CPU % by service type (last 7 days):


C:\Users\shareuser\AppData\Local\Temp\ipykernel_27980\1088442681.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(q3, conn)


,service_type,avg_cpu_percent,sample_count
0,worker,52.78,1537
1,monitor,52.59,1360
2,db,52.57,1413
3,cache,52.14,1429
4,web,52.11,1434


In [6]:
# Query 4 — open critical alerts with endpoint details
q4 = """
SELECT
    a.alert_id,
    e.hostname,
    e.datacenter,
    e.service_type,
    a.category,
    a.message,
    a.created_at
FROM telemetry.alerts a
JOIN telemetry.endpoints e ON e.endpoint_id = a.endpoint_id
WHERE
    a.severity = 'critical'
    AND a.status = 'open'
ORDER BY a.created_at DESC
LIMIT 20
"""
df4 = pd.read_sql(q4, conn)
print(f"Open critical alerts (showing up to 20): {len(df4)} rows")
df4

Open critical alerts (showing up to 20): 20 rows


C:\Users\shareuser\AppData\Local\Temp\ipykernel_27980\3139976792.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql(q4, conn)


,alert_id,hostname,datacenter,service_type,category,message,created_at
0,c86c8fc0-3625-4453-8a2e-f3f824befaed,srv-03961.citi.internal,NYC1,worker,network,Load average exceeded 4x CPU count,2026-03-23 11:09:58.063609+00:00
1,37b85a13-8780-4188-b244-94e73812f771,srv-05185.citi.internal,SNG1,web,cpu,Network throughput dropped below SLA threshold,2026-03-23 10:59:08.063609+00:00
2,e96361a9-f106-4faf-89c1-f81c8d0d976a,srv-09205.citi.internal,NYC2,monitor,disk,CPU utilization exceeded 90% threshold for 5 m...,2026-03-23 10:51:38.063609+00:00
3,a937e784-5b1f-4ca1-bbdc-90b6463decc1,srv-08492.citi.internal,SNG1,worker,memory,TCP connection pool exhausted on port 5432,2026-03-23 10:32:23.063609+00:00
4,f00672ee-30f6-4547-9dcd-83ee46ddae60,srv-03158.citi.internal,LON1,monitor,network,SSL certificate expires in 7 days,2026-03-23 09:38:57.063609+00:00
5,1c6b33a1-f776-497e-9c8a-96d0327c49bb,srv-09260.citi.internal,NYC1,db,disk,Memory usage at 95% — potential OOM imminent,2026-03-23 08:52:47.063609+00:00
6,8c1496c9-3fb9-4e43-b885-bbde555de28d,srv-01438.citi.internal,SNG1,web,disk,Disk I/O latency spike detected: 450ms avg,2026-03-23 08:33:59.063609+00:00
7,2ee1ac9b-614d-486b-a44a-f13255990e90,srv-01191.citi.internal,NYC1,web,disk,Process georgenichols exited unexpectedly — re...,2026-03-23 08:05:28.063609+00:00
8,6156565c-da0d-43e8-acd7-d8a94c503f8e,srv-06141.citi.internal,NYC1,monitor,network,CPU utilization exceeded 90% threshold for 5 m...,2026-03-23 06:48:31.063609+00:00
9,f0c24621-26a8-466e-a9a1-9436cefc72c3,srv-00057.citi.internal,LON1,db,network,TCP connection pool exhausted on port 5432,2026-03-23 06:42:24.063609+00:00


In [7]:
# Query 5 — EXPLAIN ANALYZE on Query 3
explain_q = """
EXPLAIN ANALYZE
SELECT
    e.service_type,
    ROUND(AVG(m.value)::numeric, 2) AS avg_cpu_percent,
    COUNT(*) AS sample_count
FROM telemetry.metrics m
JOIN telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
WHERE
    m.metric_name = 'cpu_percent'
    AND m.recorded_at >= NOW() - INTERVAL '7 days'
GROUP BY e.service_type
ORDER BY avg_cpu_percent DESC
"""
cur = conn.cursor()
cur.execute(explain_q)
plan = cur.fetchall()
print("\n".join(row[0] for row in plan))

Sort  (cost=11636.57..11636.58 rows=5 width=45) (actual time=36.979..39.632 rows=5 loops=1)
  Sort Key: (round((avg(m.value))::numeric, 2)) DESC
  Sort Method: quicksort  Memory: 25kB
  ->  Finalize GroupAggregate  (cost=11635.18..11636.51 rows=5 width=45) (actual time=36.965..39.624 rows=5 loops=1)
        Group Key: e.service_type
        ->  Gather Merge  (cost=11635.18..11636.35 rows=10 width=45) (actual time=36.943..39.597 rows=15 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              ->  Sort  (cost=10635.16..10635.17 rows=5 width=45) (actual time=15.632..15.634 rows=5 loops=3)
                    Sort Key: e.service_type
                    Sort Method: quicksort  Memory: 25kB
                    Worker 0:  Sort Method: quicksort  Memory: 25kB
                    Worker 1:  Sort Method: quicksort  Memory: 25kB
                    ->  Partial HashAggregate  (cost=10635.05..10635.10 rows=5 width=45) (actual time=15.610..15.613 rows=5 loops=3)
   

## Key observations

- **Scan type on metrics** — did Postgres choose a sequential scan or an index scan on `metrics.metric_name`? _Fill in after running._
- **Join strategy** — hash join or nested loop? Hash join wins when both sides are large; nested loop wins when the inner side is small and indexed. _Fill in after running._
- **Actual rows vs estimated rows** — are the planner's row estimates close to reality? Large differences mean stale statistics (`ANALYZE` would help). _Fill in after running._
- **Total execution time** — note the ms here as a baseline before adding indexes. _Fill in after running._
- **Index opportunity** — a composite index on `(metric_name, recorded_at)` would likely convert the seq scan to an index scan for time-windowed metric queries. _Try it in Round 2._